In [2]:
import sqlite3
import pandas as pd
import os

# =============================================
# CONNECT TO YOUR DATABASE
# =============================================
DB_PATH = "E:/Sidra/JupyterProjects/RiyalPay-Project/riyalpay.db"

# Check if database exists
if not os.path.exists(DB_PATH):
    print(f"❌ Database not found at: {DB_PATH}")
    print("Please run 00_load_raw.py first!")
    raise SystemExit(1) 
else:
    print(f"✅ Database found at: {DB_PATH}")

conn = sqlite3.connect(DB_PATH)
print("✅ Connected to database successfully!\n")

# =============================================
# RUN ALL PROFILING CHECKS
# =============================================

print("=" * 80)
print("📊 DATA QUALITY PROFILING REPORT")
print("=" * 80)
print()

# ---------------------------------------------------------------------
# CHECK 1: Duplicate user_id rows
# ---------------------------------------------------------------------
print("🔍 CHECK 1: Duplicate user_id rows")
print("-" * 80)

# 1a: Preview (top 20)
query = """
SELECT user_id, COUNT(*) AS n_rows
FROM raw_users
GROUP BY user_id
HAVING COUNT(*) > 1
ORDER BY n_rows DESC
LIMIT 20;
"""
df_preview = pd.read_sql(query, conn)

# 1b: Accurate total count
query_total = """
SELECT COUNT(*) AS total_users_with_duplicates FROM (
    SELECT user_id FROM raw_users
    GROUP BY user_id
    HAVING COUNT(*) > 1
);
"""
df_total = pd.read_sql(query_total, conn)
total_duplicate_users = df_total.iloc[0, 0]

if total_duplicate_users > 0:
    print(f"⚠️ Total users with duplicates: {total_duplicate_users}")
    print(f"📋 Showing top {len(df_preview)}:")
    print(df_preview)
else:
    print("✅ No duplicate user_ids found!")
print()

# 1c: Total duplicate rows count
query = """
WITH dupes AS (
  SELECT user_id FROM raw_users GROUP BY user_id HAVING COUNT(*) > 1
)
SELECT COUNT(DISTINCT user_id) AS duplicated_user_ids,
       (SELECT COUNT(*) FROM raw_users WHERE user_id IN (SELECT user_id FROM dupes)) AS total_dupe_rows
FROM dupes;
"""
df = pd.read_sql(query, conn)
if len(df) > 0 and df.iloc[0, 0] > 0:
    print(f"📊 Total duplicated users: {df.iloc[0, 0]}")
    print(f"📊 Total duplicate rows: {df.iloc[0, 1]}")
else:
    print("✅ No duplicate rows found!")
print()

# ---------------------------------------------------------------------
# CHECK 2: Field-level completeness
# ---------------------------------------------------------------------
print("🔍 CHECK 2: Field-level completeness")
print("-" * 80)

query = """
SELECT
  COUNT(*) AS total_rows,
  SUM(CASE WHEN city IS NULL THEN 1 ELSE 0 END) AS null_city,
  SUM(CASE WHEN age_bracket IS NULL THEN 1 ELSE 0 END) AS null_age_bracket,
  SUM(CASE WHEN is_test_account = 'True' THEN 1 ELSE 0 END) AS test_accounts
FROM raw_users;
"""
df = pd.read_sql(query, conn)
print("📋 User table completeness:")
print(df)
print()

# Check 2b: Distinct city spellings (top 20)
query = """
SELECT city, COUNT(*) AS n
FROM raw_users
GROUP BY city
ORDER BY n DESC
LIMIT 20;
"""
df = pd.read_sql(query, conn)
print("📋 Top 20 city spellings:")
print(df)
print()

# ---------------------------------------------------------------------
# CHECK 3: Event schema comparison
# ---------------------------------------------------------------------
print("🔍 CHECK 3: Event schema comparison (pre vs post migration)")
print("-" * 80)

# ---------------------------------------------------------------------
# Part 3a: Events appearing in only ONE source (need mapping)
# ---------------------------------------------------------------------
print("📋 Events appearing in only ONE source (need mapping):")
print("-" * 80)

query1 = """
WITH combined AS (
  SELECT 'pre_migration' AS source, event AS event_name, COUNT(*) AS n
  FROM raw_events_pre_migration GROUP BY event
  UNION ALL
  SELECT 'post_migration' AS source, event_name, COUNT(*) AS n
  FROM raw_events_post_migration GROUP BY event_name
)
SELECT event_name, source, n
FROM combined
WHERE event_name IN (
  SELECT event_name FROM combined GROUP BY event_name HAVING COUNT(DISTINCT source) = 1
)
ORDER BY event_name, source;
"""

df1 = pd.read_sql(query1, conn)
if len(df1) > 0:
    print(df1.to_string(index=False))
else:
    print("✅ All events appear in both sources!")
print()

# ---------------------------------------------------------------------
# Part 3b: Full side-by-side reference (all event names)
# ---------------------------------------------------------------------
print("📋 Full side-by-side reference (all event names):")
print("-" * 80)

query2 = """
SELECT 'pre_migration' AS source, event AS event_name, COUNT(*) AS n
FROM raw_events_pre_migration GROUP BY event
UNION ALL
SELECT 'post_migration' AS source, event_name, COUNT(*) AS n
FROM raw_events_post_migration GROUP BY event_name
ORDER BY event_name, source;
"""

df2 = pd.read_sql(query2, conn)
print(df2.to_string(index=False))
print()

# ---------------------------------------------------------------------
# Part 3c: Date ranges for each export
# ---------------------------------------------------------------------
print("📋 Date ranges for each export:")
print("-" * 80)

query3 = """
SELECT 'pre_migration' AS source, MIN(ts) AS min_ts, MAX(ts) AS max_ts 
FROM raw_events_pre_migration
UNION ALL
SELECT 'post_migration' AS source, MIN(event_timestamp), MAX(event_timestamp) 
FROM raw_events_post_migration;
"""

df3 = pd.read_sql(query3, conn)
print(df3.to_string(index=False))
print()

# ---------------------------------------------------------------------
# CHECK 4: Null user_id volume
# ---------------------------------------------------------------------
print("🔍 CHECK 4: Null user_id volume (anonymous traffic)")
print("-" * 80)

query = """
SELECT 'pre_migration' AS source, 
       SUM(CASE WHEN uid IS NULL THEN 1 ELSE 0 END) AS null_user_id, 
       COUNT(*) AS total,
       ROUND(SUM(CASE WHEN uid IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pct_null
FROM raw_events_pre_migration
UNION ALL
SELECT 'post_migration', 
       SUM(CASE WHEN user_id IS NULL THEN 1 ELSE 0 END), 
       COUNT(*),
       ROUND(SUM(CASE WHEN user_id IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2)
FROM raw_events_post_migration;
"""
df = pd.read_sql(query, conn)
print("📋 Null user_id statistics:")
print(df)
print()

# ---------------------------------------------------------------------
# CHECK 5: Exact duplicate rows (BOTH exports)
# ---------------------------------------------------------------------
print("🔍 CHECK 5: Exact duplicate rows (BOTH exports)")
print("-" * 80)

# 5a: Post-migration duplicates preview
print("📋 POST-MIGRATION Duplicate Events:")
query = """
SELECT user_id, event_name, event_timestamp, COUNT(*) AS n
FROM raw_events_post_migration
WHERE user_id IS NOT NULL
GROUP BY user_id, event_name, event_timestamp
HAVING COUNT(*) > 1
ORDER BY n DESC
LIMIT 20;
"""
df = pd.read_sql(query, conn)
if len(df) > 0:
    print(f"⚠️ Showing top {len(df)} duplicate patterns:")
    print(df)
else:
    print("✅ No duplicate events found!")
print()

# 5b: Pre-migration duplicates preview
print("📋 PRE-MIGRATION Duplicate Events:")
query = """
SELECT uid AS user_id, event AS event_name, ts AS event_timestamp, COUNT(*) AS n
FROM raw_events_pre_migration
WHERE uid IS NOT NULL
GROUP BY uid, event, ts
HAVING COUNT(*) > 1
ORDER BY n DESC
LIMIT 20;
"""
df = pd.read_sql(query, conn)
if len(df) > 0:
    print(f"⚠️ Showing top {len(df)} duplicate patterns:")
    print(df)
else:
    print("✅ No duplicate events found!")
print()

# 5c: Side-by-side comparison (accurate total)
print("📋 Side-by-side duplicate group count:")
query = """
SELECT 'post_migration' AS source, COUNT(*) AS duplicate_groups FROM (
  SELECT user_id, event_name, event_timestamp FROM raw_events_post_migration
  WHERE user_id IS NOT NULL GROUP BY user_id, event_name, event_timestamp HAVING COUNT(*) > 1
)
UNION ALL
SELECT 'pre_migration', COUNT(*) FROM (
  SELECT uid, event, ts FROM raw_events_pre_migration
  WHERE uid IS NOT NULL GROUP BY uid, event, ts HAVING COUNT(*) > 1
);
"""
df = pd.read_sql(query, conn)
print(df)
print()

# ---------------------------------------------------------------------
# CHECK 6: Suspiciously high-frequency users (bots) - BOTH exports
# ---------------------------------------------------------------------
print("🔍 CHECK 6: Suspiciously high-frequency users (potential bots)")
print("-" * 80)

# 6a: Accurate total count for post-migration
query_total = """
SELECT 'post_migration' AS source, COUNT(*) AS n_users_over_200_events FROM (
  SELECT user_id FROM raw_events_post_migration WHERE user_id IS NOT NULL
  GROUP BY user_id HAVING COUNT(*) > 200
)
UNION ALL
SELECT 'pre_migration', COUNT(*) FROM (
  SELECT uid FROM raw_events_pre_migration WHERE uid IS NOT NULL
  GROUP BY uid HAVING COUNT(*) > 200
);
"""
df_total = pd.read_sql(query_total, conn)
print("📋 Total users with >200 events:")
print(df_total)
print()

# 6b: Post-migration bots preview (top 30 by events_per_day)
print("📋 POST-MIGRATION - Top 30 users by events_per_day:")
query = """
SELECT user_id, COUNT(*) AS n_events,
       ROUND(COUNT(*) * 1.0 / (JULIANDAY(MAX(event_timestamp)) - JULIANDAY(MIN(event_timestamp)) + 1), 2) AS events_per_day
FROM raw_events_post_migration
WHERE user_id IS NOT NULL
GROUP BY user_id
HAVING n_events > 200
ORDER BY events_per_day DESC
LIMIT 30;
"""
df = pd.read_sql(query, conn)
if len(df) > 0:
    print(f"⚠️ Showing top {len(df)} users (highest events/day):")
    print(df)
else:
    print("✅ No suspicious high-frequency users found!")
print()

# 6c: Pre-migration bots preview (top 30 by events_per_day)
print("📋 PRE-MIGRATION - Top 30 users by events_per_day:")
query = """
SELECT uid AS user_id, COUNT(*) AS n_events,
       ROUND(COUNT(*) * 1.0 / (JULIANDAY(MAX(ts)) - JULIANDAY(MIN(ts)) + 1), 2) AS events_per_day
FROM raw_events_pre_migration
WHERE uid IS NOT NULL
GROUP BY uid
HAVING n_events > 200
ORDER BY events_per_day DESC
LIMIT 30;
"""
df = pd.read_sql(query, conn)
if len(df) > 0:
    print(f"⚠️ Showing top {len(df)} users (highest events/day):")
    print(df)
else:
    print("✅ No suspicious high-frequency users found!")
print()

# ---------------------------------------------------------------------
# CHECK 7: Late-arriving data (BOTH exports)
# ---------------------------------------------------------------------
print("🔍 CHECK 7: Late-arriving data (ingested_at vs event_timestamp)")
print("-" * 80)

# 7a: Side-by-side accurate percentage
query = """
SELECT 'post_migration' AS source,
       ROUND(SUM(CASE WHEN JULIANDAY(ingested_at) - JULIANDAY(event_timestamp) > 0.04 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pct_late
FROM raw_events_post_migration
UNION ALL
SELECT 'pre_migration',
       ROUND(SUM(CASE WHEN JULIANDAY(ingested_at) - JULIANDAY(ts) > 0.04 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2)
FROM raw_events_pre_migration;
"""
df = pd.read_sql(query, conn)
print("📋 Percentage of late-arriving events:")
print(df)
print()

# 7b: Post-migration late data buckets (top 20)
print("📋 POST-MIGRATION - Late data buckets:")
query = """
SELECT
  CAST((JULIANDAY(ingested_at) - JULIANDAY(event_timestamp)) * 24 AS INT) AS hours_late_bucket,
  COUNT(*) AS n
FROM raw_events_post_migration
WHERE JULIANDAY(ingested_at) - JULIANDAY(event_timestamp) > 0.04
GROUP BY hours_late_bucket
ORDER BY hours_late_bucket DESC
LIMIT 20;
"""
df = pd.read_sql(query, conn)
if len(df) > 0:
    print("📋 Top 20 hour buckets (hours delayed):")
    print(df)
else:
    print("✅ No significant late-arriving data found!")
print()

# 7c: Pre-migration late data buckets (top 20)
print("📋 PRE-MIGRATION - Late data buckets:")
query = """
SELECT
  CAST((JULIANDAY(ingested_at) - JULIANDAY(ts)) * 24 AS INT) AS hours_late_bucket,
  COUNT(*) AS n
FROM raw_events_pre_migration
WHERE JULIANDAY(ingested_at) - JULIANDAY(ts) > 0.04
GROUP BY hours_late_bucket
ORDER BY hours_late_bucket DESC
LIMIT 20;
"""
df = pd.read_sql(query, conn)
if len(df) > 0:
    print("📋 Top 20 hour buckets (hours delayed):")
    print(df)
else:
    print("✅ No significant late-arriving data found!")
print()


# ---------------------------------------------------------------------
# CHECK 8: Missing app_version (with breakdown)
# ---------------------------------------------------------------------
print("🔍 CHECK 8: Missing app_version (detailed breakdown)")
print("-" * 80)

# 8a: Post-migration detailed breakdown
print("📋 POST-MIGRATION - app_version breakdown:")
query = """
SELECT
  CASE WHEN app_version IS NULL THEN 'null'
       WHEN LOWER(app_version) = 'unknown' THEN 'unknown'
       ELSE 'has_version' END AS version_status,
  CASE WHEN user_id IS NULL THEN 'anonymous' ELSE 'identified' END AS user_status,
  COUNT(*) AS n,
  ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM raw_events_post_migration), 2) AS pct
FROM raw_events_post_migration
GROUP BY version_status, user_status
ORDER BY version_status, user_status;
"""
df = pd.read_sql(query, conn)
print(df)
print()

# 8b: Pre-migration detailed breakdown
print("📋 PRE-MIGRATION - app_version breakdown:")
query = """
SELECT
  CASE WHEN ver IS NULL THEN 'null'
       WHEN LOWER(ver) = 'unknown' THEN 'unknown'
       ELSE 'has_version' END AS version_status,
  CASE WHEN uid IS NULL THEN 'anonymous' ELSE 'identified' END AS user_status,
  COUNT(*) AS n,
  ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM raw_events_pre_migration), 2) AS pct
FROM raw_events_pre_migration
GROUP BY version_status, user_status
ORDER BY version_status, user_status;
"""
df = pd.read_sql(query, conn)
print(df)
print()

# 8c: The REAL metric - genuine gap (identified users with NULL)
print("📋 GENUINE GAP (identified users with NULL version):")
query = """
SELECT 'post_migration' AS source,
       ROUND(SUM(CASE WHEN app_version IS NULL AND user_id IS NOT NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 4) AS pct_genuine_gap
FROM raw_events_post_migration
UNION ALL
SELECT 'pre_migration',
       ROUND(SUM(CASE WHEN ver IS NULL AND uid IS NOT NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 4)
FROM raw_events_pre_migration;
"""
df = pd.read_sql(query, conn)
print(df)
print()

# 8d: Raw counts for genuine gap
print("📋 GENUINE GAP - raw counts:")
query = """
SELECT 'post_migration' AS source,
       SUM(CASE WHEN app_version IS NULL AND user_id IS NOT NULL THEN 1 ELSE 0 END) AS genuine_gap_events,
       COUNT(*) AS total_events
FROM raw_events_post_migration
UNION ALL
SELECT 'pre_migration',
       SUM(CASE WHEN ver IS NULL AND uid IS NOT NULL THEN 1 ELSE 0 END),
       COUNT(*)
FROM raw_events_pre_migration;
"""
df = pd.read_sql(query, conn)
print(df)
print()

# ---------------------------------------------------------------------
# CHECK 9: Currency field messiness
# ---------------------------------------------------------------------
print("🔍 CHECK 9: Currency field values")
print("-" * 80)

query = """
SELECT currency, COUNT(*) AS n
FROM raw_transactions
GROUP BY currency
ORDER BY n DESC;
"""
df = pd.read_sql(query, conn)
print("📋 Currency distribution:")
print(df)
print()

# ---------------------------------------------------------------------
# CHECK 10: Transactions with missing event link
# ---------------------------------------------------------------------
print("🔍 CHECK 10: Transactions with missing event link")
print("-" * 80)

query = """
SELECT status, transaction_type,
       SUM(CASE WHEN linked_event_id IS NULL THEN 1 ELSE 0 END) AS missing_event_link,
       COUNT(*) AS total,
       ROUND(SUM(CASE WHEN linked_event_id IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pct_missing
FROM raw_transactions
GROUP BY status, transaction_type
ORDER BY total DESC;
"""
df = pd.read_sql(query, conn)
print("📋 Transaction-to-event link gaps:")
print(df)
print()

# ---------------------------------------------------------------------
# CHECK 11: Outage-day anomaly
# ---------------------------------------------------------------------
print("🔍 CHECK 11: Payment outage days (high failure rates)")
print("-" * 80)

query = """
WITH daily AS (
  SELECT DATE(transaction_timestamp) AS txn_date,
         SUM(CASE WHEN status='failed' THEN 1 ELSE 0 END) AS n_failed,
         COUNT(*) AS n_total,
         ROUND(SUM(CASE WHEN status='failed' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS failure_rate_pct
  FROM raw_transactions
  WHERE transaction_type = 'top_up'
  GROUP BY txn_date
)
SELECT txn_date, n_failed, n_total, failure_rate_pct
FROM daily
WHERE failure_rate_pct > 15
ORDER BY failure_rate_pct DESC;
"""
df = pd.read_sql(query, conn)
if len(df) > 0:
    print(f"⚠️ Found {len(df)} days with >15% failure rate:")
    print(df)
else:
    print("✅ No outage days detected (>15% failure rate)")
print()

# ---------------------------------------------------------------------
# CHECK 12: Experiment contamination
# ---------------------------------------------------------------------
print("🔍 CHECK 12: Experiment assignment contamination")
print("-" * 80)

# 12a: Accurate total
query_total = """
SELECT COUNT(*) AS n_users_with_duplicate_assignments FROM (
  SELECT user_id FROM raw_experiment_assignments
  GROUP BY user_id HAVING COUNT(*) > 1
);
"""
df_total = pd.read_sql(query_total, conn)
total_contaminated = df_total.iloc[0, 0]
print(f"📋 Total users with duplicate assignments: {total_contaminated}")
print()

# 12b: Preview (top 20)
query = """
SELECT user_id, COUNT(*) AS n_assignments, COUNT(DISTINCT variant) AS n_distinct_variants
FROM raw_experiment_assignments
GROUP BY user_id
HAVING COUNT(*) > 1
ORDER BY n_distinct_variants DESC, n_assignments DESC
LIMIT 20;
"""
df = pd.read_sql(query, conn)
if len(df) > 0:
    print(f"⚠️ Showing top {len(df)} users with multiple assignments:")
    print(df)
    contaminated = df[df['n_distinct_variants'] > 1]
    if len(contaminated) > 0:
        print(f"⚠️ {len(contaminated)} users assigned to BOTH control and test!")
    else:
        print("✅ No users assigned to both arms (but some have duplicate assignments)")
else:
    print("✅ No assignment contamination found!")
print()

# ---------------------------------------------------------------------
# CHECK 13: Sample ratio check
# ---------------------------------------------------------------------
print("🔍 CHECK 13: Sample ratio check by platform (50/50 split?)")
print("-" * 80)

query = """
SELECT platform, variant, COUNT(DISTINCT user_id) AS n_users
FROM raw_experiment_assignments
GROUP BY platform, variant
ORDER BY platform, variant;
"""
df = pd.read_sql(query, conn)
print("📋 Experiment sample ratio:")
print(df)
print()

# Calculate ratios
if len(df) > 0:
    for platform in df['platform'].unique():
        platform_data = df[df['platform'] == platform]
        variants = platform_data['variant'].unique()
        
        if len(variants) == 2:
            variant1 = variants[0]
            variant2 = variants[1]
            n1 = platform_data[platform_data['variant'] == variant1]['n_users'].values[0]
            n2 = platform_data[platform_data['variant'] == variant2]['n_users'].values[0]
            total = n1 + n2
            pct1 = round(n1 * 100 / total, 1)
            pct2 = round(n2 * 100 / total, 1)
            print(f"📊 {platform}: {variant1}={pct1}%, {variant2}={pct2}%")
            if abs(pct1 - pct2) > 5:
                print(f"   ⚠️  WARNING: More than 5% difference from 50/50!")
            else:
                print(f"   ✅ Good: Close to 50/50 split")
        elif len(variants) == 1:
            variant = variants[0]
            n = platform_data['n_users'].values[0]
            print(f"📊 {platform}: Only {variant} found ({n} users)")
            print(f"   ⚠️  WARNING: Missing the other variant!")
        else:
            print(f"📊 {platform}: {len(variants)} variants found:")
            for variant in variants:
                n = platform_data[platform_data['variant'] == variant]['n_users'].values[0]
                print(f"   - {variant}: {n} users")
else:
    print("⚠️ No experiment assignment data found!")
print()

# ---------------------------------------------------------------------
# CHECK 14: Events dated BEFORE the user's signup (temporal integrity)
# ---------------------------------------------------------------------
print("🔍 CHECK 14: Events before signup (temporal integrity)")
print("-" * 80)

# 14a: Accurate totals (no LIMIT)
query = """
WITH u AS (SELECT user_id, MIN(signup_date) AS signup_date FROM raw_users GROUP BY user_id)
SELECT 'post_migration' AS source, COUNT(*) AS events_before_signup, COUNT(DISTINCT e.user_id) AS users_affected
FROM raw_events_post_migration e JOIN u ON u.user_id = e.user_id
WHERE JULIANDAY(e.event_timestamp) < JULIANDAY(u.signup_date) - 1
UNION ALL
SELECT 'pre_migration', COUNT(*), COUNT(DISTINCT e.uid)
FROM raw_events_pre_migration e JOIN u ON u.user_id = e.uid
WHERE JULIANDAY(e.ts) < JULIANDAY(u.signup_date) - 1;
"""
df = pd.read_sql(query, conn)
print("📋 Accurate totals (events before signup):")
print(df.to_string(index=False))
print()

# 14b: Preview of offenders (top 20)
query = """
WITH u AS (SELECT user_id, MIN(signup_date) AS signup_date FROM raw_users GROUP BY user_id)
SELECT e.user_id, u.signup_date, COUNT(*) AS events_before_signup,
       MIN(e.event_timestamp) AS earliest_event,
       COUNT(DISTINCT SUBSTR(e.event_timestamp, 1, 13)) AS distinct_hours
FROM raw_events_post_migration e JOIN u ON u.user_id = e.user_id
WHERE JULIANDAY(e.event_timestamp) < JULIANDAY(u.signup_date) - 1
GROUP BY e.user_id, u.signup_date
ORDER BY events_before_signup DESC
LIMIT 20;
"""
df = pd.read_sql(query, conn)
if len(df) > 0:
    print(f"⚠️ Showing top {len(df)} offenders (events before signup):")
    print(df.to_string(index=False))
    print("   Note: distinct_hours=1 with hundreds of events is the burst signature.")
else:
    print("✅ No events found before signup!")
print()

# ---------------------------------------------------------------------
# CHECK 15: Burst detection -- peak events per user per hour
# ---------------------------------------------------------------------
print("🔍 CHECK 15: Burst detection (peak events per user per hour)")
print("-" * 80)

# 15a: Accurate totals by ID type
query = """
WITH hourly AS (
  SELECT user_id, SUBSTR(event_timestamp, 1, 13) AS hour_bucket, COUNT(*) AS n
  FROM raw_events_post_migration WHERE user_id IS NOT NULL
  GROUP BY user_id, hour_bucket HAVING COUNT(*) > 100
),
u AS (SELECT DISTINCT user_id FROM raw_users)
SELECT CASE WHEN u.user_id IS NULL THEN 'no user row (pure bot)' ELSE 'real user ID (collision)' END AS id_type,
       COUNT(DISTINCT h.user_id) AS user_ids,
       COUNT(*) AS burst_hours,
       SUM(h.n) AS events_in_burst_hours
FROM hourly h LEFT JOIN u ON u.user_id = h.user_id
GROUP BY id_type;
"""
df = pd.read_sql(query, conn)
print("📋 Burst summary (post-migration):")
print(df.to_string(index=False))
print()

# 15b: Pre-migration burst hours (expect none)
query = """
SELECT COUNT(*) AS pre_migration_burst_hours FROM (
  SELECT uid, SUBSTR(ts, 1, 13) AS hour_bucket
  FROM raw_events_pre_migration WHERE uid IS NOT NULL
  GROUP BY uid, hour_bucket HAVING COUNT(*) > 100
);
"""
df = pd.read_sql(query, conn)
print("📋 Pre-migration burst hours:")
print(df.to_string(index=False))
print()

# 15c: Preview of heaviest bursts (top 15)
query = """
SELECT user_id, SUBSTR(event_timestamp, 1, 13) AS hour_bucket, COUNT(*) AS events_in_hour
FROM raw_events_post_migration WHERE user_id IS NOT NULL
GROUP BY user_id, hour_bucket HAVING COUNT(*) > 100
ORDER BY events_in_hour DESC
LIMIT 15;
"""
df = pd.read_sql(query, conn)
if len(df) > 0:
    print(f"⚠️ Showing top {len(df)} heaviest burst hours:")
    print(df.to_string(index=False))
else:
    print("✅ No burst hours detected!")
print()

# ---------------------------------------------------------------------
# CHECK 16: Funnel integrity -- later-stage events without earlier ones
# ---------------------------------------------------------------------
print("🔍 CHECK 16: Funnel integrity (later-stage events without earlier ones)")
print("-" * 80)

# 16a: Funnel integrity violations (combining both exports)
query = """
WITH ev AS (
  SELECT uid AS user_id, event AS event_name FROM raw_events_pre_migration WHERE uid IS NOT NULL
  UNION ALL
  SELECT user_id, event_name FROM raw_events_post_migration WHERE user_id IS NOT NULL
),
per_user AS (
  SELECT user_id,
         MAX(event_name = 'signup_complete') AS has_signup,
         MAX(event_name = 'signup_start')    AS has_signup_start,
         MAX(event_name = 'kyc_start')       AS has_kyc_start,
         MAX(event_name = 'kyc_approved')    AS has_kyc_approved
  FROM ev GROUP BY user_id
)
SELECT
  SUM(CASE WHEN has_kyc_start = 1 AND has_signup = 0 THEN 1 ELSE 0 END)      AS kyc_start_without_signup_complete,
  SUM(CASE WHEN has_kyc_start = 1 AND has_signup_start = 0 THEN 1 ELSE 0 END) AS kyc_start_without_signup_start,
  SUM(CASE WHEN has_kyc_approved = 1 AND has_kyc_start = 0 THEN 1 ELSE 0 END) AS approved_without_kyc_start,
  SUM(CASE WHEN has_kyc_start = 1 THEN 1 ELSE 0 END)                         AS users_with_kyc_start
FROM per_user;
"""
df = pd.read_sql(query, conn)
print("📋 Funnel integrity violations:")
print(df.to_string(index=False))
print()

# 16b: Successful transactions by users with no kyc_approved event
query = """
SELECT COUNT(DISTINCT t.user_id) AS txn_users_without_kyc_approved
FROM raw_transactions t
WHERE t.status = 'success'
  AND t.user_id NOT IN (
    SELECT uid FROM raw_events_pre_migration WHERE event = 'kyc_approved' AND uid IS NOT NULL
    UNION
    SELECT user_id FROM raw_events_post_migration WHERE event_name = 'kyc_approved' AND user_id IS NOT NULL
  );
"""
df = pd.read_sql(query, conn)
print("📋 Successful transactions without kyc_approved event:")
print(df.to_string(index=False))
print()
# =============================================
# CLOSE CONNECTION
# =============================================
conn.close()

print()
print("=" * 80)
print("✅ PROFILING COMPLETE!")
print("=" * 80)
print()
print("📝 Summary of findings:")
print("   Review the outputs above for data quality issues.")
print("   These will guide your cleaning in the next step (02_reconcile_and_clean.py)")

✅ Database found at: E:/Sidra/JupyterProjects/RiyalPay-Project/riyalpay.db
✅ Connected to database successfully!

📊 DATA QUALITY PROFILING REPORT

🔍 CHECK 1: Duplicate user_id rows
--------------------------------------------------------------------------------
⚠️ Total users with duplicates: 336
📋 Showing top 20:
    user_id  n_rows
0   U127959       2
1   U127866       2
2   U127850       2
3   U127763       2
4   U127723       2
5   U127702       2
6   U127526       2
7   U127515       2
8   U127153       2
9   U126887       2
10  U126739       2
11  U126599       2
12  U126557       2
13  U126524       2
14  U126452       2
15  U126400       2
16  U126319       2
17  U126202       2
18  U126192       2
19  U126126       2

📊 Total duplicated users: 336
📊 Total duplicate rows: 672

🔍 CHECK 2: Field-level completeness
--------------------------------------------------------------------------------
📋 User table completeness:
   total_rows  null_city  null_age_bracket  test_accounts
0 